# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a walkthrough for loading and exploring the [FAIR²](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Explore available record sets, their fields, and identify their `@id`s.

**Note:** All references to Record Sets, Fields, and Columns use their `@id` attribute.

In [ ]:
# List available record sets and their fields
print("Available record sets and their fields:")
record_set_ids = []
for rset in dataset.metadata.record_set or []:
    print(f"\nRecordSet name: {rset.name}")
    print(f"  @id : {rset['@id']}")
    record_set_ids.append(rset['@id'])
    if hasattr(rset, 'field') and rset.field:
        print("  Fields:")
        for field in rset.field:
            print(f"    - {field.name} (@id: {field['@id']}) Type: {field.data_type if hasattr(field, 'data_type') else '?'}")
    else:
        print("  No fields listed.")
if not record_set_ids:
    print("No record sets found in schema metadata. Attempting to check 'hasPart' for data-oriented entries.")
    # Try to infer possible record sets from hasPart
    if hasattr(dataset.metadata, 'has_part') and dataset.metadata.has_part:
        for part in dataset.metadata.has_part:
            if 'RecordSet' in part.get('@type', ''):
                print(f"Possible record set: {part.get('name', '?')} (@id: {part.get('@id')})")
                record_set_ids.append(part['@id'])
if not record_set_ids:
    print("No record sets identified. Please check the Croissant schema for available data.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

If the dataset does not define explicit record sets, extract the first available record set or try to load records using `None` as the `record_set` parameter.

In [ ]:
# Try to infer the main record set for extraction
if record_set_ids:
    selected_record_set_id = record_set_ids[0]
    print(f"Using record set: {selected_record_set_id}")
else:
    selected_record_set_id = None
    print("No explicit record sets found; will attempt to load all records as a fallback.")

try:
    records = list(dataset.records(record_set=selected_record_set_id))
except Exception as exc:
    print(f"Failed to load records with 'record_set' argument: {exc}")
    print("Attempting to load all records...")
    records = list(dataset.records())

df = pd.DataFrame(records)
print(f"Columns available in the main DataFrame:\n{df.columns.tolist()}")
df.head()

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing: filter numeric fields, normalize, and group if applicable.

Below, replace the field `@id` with the appropriate key as determined above. Here we attempt to select a numeric field, such as 'MW [kDa]', 'Norm Abundance', 'Unique Peptides', etc.

In [ ]:
# Identify a likely numeric field
import numpy as np

potential_numeric_fields = [c for c in df.columns if any(x in c.lower() for x in ['abundance', 'mw', 'unique', 'coverage', 'peptide', 'score', 'count'])]
if potential_numeric_fields:
    numeric_field = potential_numeric_fields[0]
    print(f"Using numeric field for filtering/analysis: {numeric_field}")
else:
    print("No obvious numeric field found. Using first column for demonstration.")
    numeric_field = df.columns[0]

# Try to filter values > median for visualization; fallback threshold is 0
if np.issubdtype(df[numeric_field].dtype, np.number):
    threshold = df[numeric_field].median() if df[numeric_field].notnull().any() else 0
else:
    try:
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].median() if df[numeric_field].notnull().any() else 0
    except:
        threshold = 0

filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (median):")
print(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() != 0 else 1)
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to identify a grouping field, e.g., sample group or description
group_field_candidates = [x for x in df.columns if any(gk in x.lower() for gk in ['group', 'sample', 'category', 'class', 'type'])]
group_field = group_field_candidates[0] if group_field_candidates else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())
else:
    print("No grouping field found.")

## 5. Visualization
Visualize distribution of the selected numeric field and its normalized version.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 5))
sns.histplot(filtered_df[numeric_field].dropna(), bins=30, kde=True, color='skyblue')
plt.title(f"Histogram of filtered {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel('Frequency')
plt.show()

plt.figure(figsize=(12, 5))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True, color='coral')
plt.title(f"Histogram of Normalized {numeric_field}")
plt.xlabel(f"{numeric_field} (normalized)")
plt.ylabel('Frequency')
plt.show()

# If group_field exists, visualize mean numeric field by group
if group_field:
    plt.figure(figsize=(12, 6))
    grouped = filtered_df.groupby(group_field)[numeric_field].mean().sort_values()
    grouped.plot(kind='bar', color='mediumpurple')
    plt.title(f"Mean {numeric_field} by {group_field}")
    plt.ylabel(f"Mean {numeric_field}")
    plt.xlabel(group_field)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we demonstrated how to use the `mlcroissant` library to:

- Load mass spectrometry protein data defined in a FAIR² / Croissant schema
- Discover and inspect record sets, fields, and their unique `@id` references
- Load and manipulate tabular data from the dataset
- Filter and normalize a numeric measurement field, and visualize distributions

For further analysis, you can select other fields (using their `@id` or DataFrame column key), join across record sets if present, or explore relationships and metadata enrichment provided in the schema. Refer to the [mlcroissant documentation](https://mlcommons.org/croissant/) for advanced use cases.